In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
import os
os.makedirs('images/open_transaction', exist_ok=True)


In [2]:
# Load original raw Open Transaction dataset
file_path = 'property data/Open Transaction Data.xlsx'
sheet_name = 'Open Transaction Data'

# ensure path exists
import os
if not os.path.exists(file_path):
    raise FileNotFoundError(f"Data file not found: {file_path}")
df_raw = pd.read_excel(file_path, sheet_name=sheet_name)

print(f'Raw dataset loaded successfully.')
print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
df_raw.head()


Raw dataset loaded successfully.
Shape: 416,627 rows x 13 columns


,Property Type,District,Mukim,Scheme Name/Area,Road Name,"Month, Year of Transaction Date",Tenure,Land/Parcel Area,Unit,Main Floor Area,Unit,Unit Level,Transaction Price
0,1 - 1 1/2 Storey Semi-Detached,Alor Gajah,Bdr Masjid Tanah,TMN BUKIT INDAH FASA 2,JLN BUKIT INDAH 6,2024-03-01,Freehold,374.0,sq.m,106,sq.m,,"RM470,000.00"
1,1 - 1 1/2 Storey Semi-Detached,Alor Gajah,Belimbing,TMN BELIMBING HARMONI,JLN BELIMBING HARMONI,2023-05-01,Leasehold,234.0,sq.m,67,sq.m,,"RM483,000.00"
2,1 - 1 1/2 Storey Semi-Detached,Alor Gajah,Belimbing,TMN VISTA BELIMBING,JALAN DURIAN TUNGGAL-ALOR GAJAH,2021-04-01,Freehold,268.0,sq.m,118,sq.m,,"RM359,000.00"
3,1 - 1 1/2 Storey Semi-Detached,Alor Gajah,Belimbing,TMN VISTA BELIMBING,JALAN DURIAN TUNGGAL-ALOR GAJAH,2021-04-01,Freehold,268.0,sq.m,118,sq.m,,"RM435,000.00"
4,1 - 1 1/2 Storey Semi-Detached,Alor Gajah,Belimbing,TMN VISTA BELIMBING,JALAN DURIAN TUNGGAL-ALOR GAJAH,2021-04-01,Freehold,297.0,sq.m,118,sq.m,,"RM373,000.00"


## Raw Open Transaction data: landed vs high-rise area checks

Purpose: verify the relationship directly from `property data/Open Transaction Data.xlsx` before changing preprocessing.

Working interpretation for the raw file:
- `Land/Parcel Area` = raw land / parcel area column.
- `Main Floor Area` = raw main floor area column.
- High-rise / non-landed types: `Condominium/Apartment`, `Flat`, `Low-Cost Flat`.
- Landed types: all other property types in this file, including `Town House` unless treated separately later.
- Raw missing values include both actual blanks and dash placeholders like `-`.


In [3]:
# Clean duplicate/trailing-space column names from the raw workbook
df_check = df_raw.copy()

clean_cols = []
seen = {}
for col in df_check.columns:
    base = str(col).strip()
    seen[base] = seen.get(base, 0) + 1
    clean_cols.append(base if seen[base] == 1 else f'{base}_{seen[base]}')
df_check.columns = clean_cols

land_col = 'Land/Parcel Area'
main_floor_col = 'Main Floor Area'
high_rise_types = ['Condominium/Apartment', 'Flat', 'Low-Cost Flat']

def is_blank_or_dash(series):
    as_text = series.astype('string').str.strip()
    return series.isna() | as_text.isin(['', '-', 'nan', 'NaN', 'None'])

df_check['is_high_rise'] = df_check['Property Type'].isin(high_rise_types)
df_check['is_landed'] = (~df_check['is_high_rise']).astype(int)

print('Cleaned columns:')
print(df_check.columns.tolist())
print('\nProperty type counts:')
display(df_check['Property Type'].value_counts().rename_axis('Property Type').reset_index(name='rows'))


Cleaned columns:
['Property Type', 'District', 'Mukim', 'Scheme Name/Area', 'Road Name', 'Month, Year of Transaction Date', 'Tenure', 'Land/Parcel Area', 'Unit', 'Main Floor Area', 'Unit_2', 'Unit Level', 'Transaction Price', 'is_high_rise', 'is_landed']

Property type counts:


,Property Type,rows
0,2 - 2 1/2 Storey Terraced,118722
1,1 - 1 1/2 Storey Terraced,93697
2,Condominium/Apartment,65780
3,Low-Cost House,28351
4,1 - 1 1/2 Storey Semi-Detached,22030
5,Low-Cost Flat,21190
6,Detached,18939
7,2 - 2 1/2 Storey Semi-Detached,18071
8,Flat,16504
9,Cluster House,9092


In [4]:
# Completeness summary by property type from the raw workbook
def completeness(group):
    land_missing = is_blank_or_dash(group[land_col])
    main_missing = is_blank_or_dash(group[main_floor_col])
    return pd.Series({
        'rows': len(group),
        'is_landed': int(group['is_landed'].max()),
        'land_missing_or_dash': int(land_missing.sum()),
        'land_present': int((~land_missing).sum()),
        'land_unique_values': int(group.loc[~land_missing, land_col].nunique(dropna=True)),
        'main_floor_missing_or_dash': int(main_missing.sum()),
        'main_floor_present': int((~main_missing).sum()),
        'main_floor_unique_values': int(group.loc[~main_missing, main_floor_col].nunique(dropna=True)),
    })

raw_area_summary = (
    df_check
    .groupby('Property Type', dropna=False)
    .apply(completeness, include_groups=False)
    .reset_index()
)
raw_area_summary['land_missing_pct'] = (raw_area_summary['land_missing_or_dash'] / raw_area_summary['rows'] * 100).round(2)
raw_area_summary['main_floor_missing_pct'] = (raw_area_summary['main_floor_missing_or_dash'] / raw_area_summary['rows'] * 100).round(2)

raw_area_summary.sort_values(['is_landed', 'Property Type'], ascending=[False, True])


,Property Type,rows,is_landed,land_missing_or_dash,land_present,land_unique_values,main_floor_missing_or_dash,main_floor_present,main_floor_unique_values,land_missing_pct,main_floor_missing_pct
0,1 - 1 1/2 Storey Semi-Detached,22030,1,0,22030,3093,3,22027,246,0.00,0.01
1,1 - 1 1/2 Storey Terraced,93697,1,0,93697,4671,9,93688,226,0.00,0.01
2,2 - 2 1/2 Storey Semi-Detached,18071,1,0,18071,3343,2,18069,446,0.00,0.01
3,2 - 2 1/2 Storey Terraced,118722,1,0,118722,6278,0,118722,426,0.00,0.00
4,Cluster House,9092,1,0,9092,1064,1,9091,273,0.00,0.01
6,Detached,18939,1,1,18938,4749,0,18939,803,0.01,0.00
9,Low-Cost House,28351,1,0,28351,1571,0,28351,168,0.00,0.00
10,Town House,4251,1,0,4251,588,4251,0,0,0.00,100.00
5,Condominium/Apartment,65780,0,0,65780,2885,65780,0,0,0.00,100.00
7,Flat,16504,0,0,16504,851,16504,0,0,0.00,100.00


In [5]:
# Check whether high-rise Land/Parcel Area is a constant or varying real values
high_rise_mask = df_check['is_high_rise']
landed_mask = df_check['is_landed'].eq(1)
high_rise = df_check.loc[high_rise_mask].copy()
high_rise_land_numeric = pd.to_numeric(high_rise[land_col], errors='coerce')

direct_answers = pd.DataFrame([
    {
        'question': 'Do all landed properties have Land/Parcel Area?',
        'rows_checked': int(landed_mask.sum()),
        'missing_or_dash_rows': int(is_blank_or_dash(df_check.loc[landed_mask, land_col]).sum()),
        'answer': 'YES' if (~is_blank_or_dash(df_check.loc[landed_mask, land_col])).all() else 'NO',
    },
    {
        'question': 'Do all high-rise properties have empty Land/Parcel Area?',
        'rows_checked': int(high_rise_mask.sum()),
        'missing_or_dash_rows': int(is_blank_or_dash(df_check.loc[high_rise_mask, land_col]).sum()),
        'answer': 'YES' if is_blank_or_dash(df_check.loc[high_rise_mask, land_col]).all() else 'NO',
    },
    {
        'question': 'Do all transactions have Main Floor Area?',
        'rows_checked': int(len(df_check)),
        'missing_or_dash_rows': int(is_blank_or_dash(df_check[main_floor_col]).sum()),
        'answer': 'YES' if (~is_blank_or_dash(df_check[main_floor_col])).all() else 'NO',
    },
])

high_rise_land_stats = pd.DataFrame([{
    'high_rise_rows': len(high_rise),
    'land_missing_or_dash': int(is_blank_or_dash(high_rise[land_col]).sum()),
    'land_unique_numeric_values': int(high_rise_land_numeric.nunique(dropna=True)),
    'land_min': high_rise_land_numeric.min(),
    'land_median': high_rise_land_numeric.median(),
    'land_max': high_rise_land_numeric.max(),
    'main_floor_missing_or_dash': int(is_blank_or_dash(high_rise[main_floor_col]).sum()),
}])

print('Direct answers:')
display(direct_answers)

print('High-rise Land/Parcel Area distribution summary:')
display(high_rise_land_stats)

print('Top high-rise Land/Parcel Area values:')
display(high_rise_land_numeric.value_counts().head(15).rename_axis('Land/Parcel Area').reset_index(name='rows'))


Direct answers:


,question,rows_checked,missing_or_dash_rows,answer
0,Do all landed properties have Land/Parcel Area?,313153,1,NO
1,Do all high-rise properties have empty Land/Pa...,103474,0,NO
2,Do all transactions have Main Floor Area?,416627,107740,NO


High-rise Land/Parcel Area distribution summary:


,high_rise_rows,land_missing_or_dash,land_unique_numeric_values,land_min,land_median,land_max,main_floor_missing_or_dash
0,103474,0,3408,12.0,79.0,5202.0,103474


Top high-rise Land/Parcel Area values:


,Land/Parcel Area,rows
0,61.0,3360
1,60.0,2656
2,65.0,2644
3,79.0,2474
4,70.0,2152
5,93.0,1785
6,84.0,1731
7,62.0,1466
8,64.0,1415
9,75.0,1352


In [8]:
# show the index and full row for the only landed property missing Land/Parcel Area
print("Missing landed Land/Parcel Area index:", landed_missing_land.index.tolist())
display(landed_missing_land)

Missing landed Land/Parcel Area index: [339142]


,Property Type,District,Mukim,Scheme Name/Area,Road Name,"Month, Year of Transaction Date",Tenure,Land/Parcel Area,Unit,Main Floor Area,Unit_2,Unit Level,Transaction Price
339142,Detached,Larut Matang,Asam Kumbang,KAMUNTING BARU TAMBAHAN (EXPO),LORONG EXPO 2,2025-10-01,Leasehold,NaN,sq.m,174,sq.m,,"RM400,000.00"


In [6]:
# Exception rows and samples from the raw workbook
landed_missing_land = df_check.loc[landed_mask & is_blank_or_dash(df_check[land_col]), [
    'Property Type', 'District', 'Mukim', 'Scheme Name/Area', 'Road Name',
    'Month, Year of Transaction Date', 'Tenure', land_col, 'Unit',
    main_floor_col, 'Unit_2', 'Unit Level', 'Transaction Price'
]]

missing_main_floor_by_type = (
    df_check[is_blank_or_dash(df_check[main_floor_col])]
    .groupby(['is_landed', 'Property Type'], dropna=False)
    .size()
    .reset_index(name='missing_main_floor_rows')
    .sort_values(['is_landed', 'missing_main_floor_rows'], ascending=[False, False])
)

high_rise_samples = high_rise[[
    'Property Type', land_col, 'Unit', main_floor_col, 'Unit_2', 'Unit Level', 'Transaction Price'
]].head(20)

print('High-rise samples: Land/Parcel Area varies; Main Floor Area is dash.')
display(high_rise_samples)

print('Landed rows missing Land/Parcel Area:')
display(landed_missing_land)

print('Rows missing/dash Main Floor Area by property type:')
display(missing_main_floor_by_type)


High-rise samples: Land/Parcel Area varies; Main Floor Area is dash.


,Property Type,Land/Parcel Area,Unit,Main Floor Area,Unit_2,Unit Level,Transaction Price
261612,Condominium/Apartment,100.00,sq.m,-,-,2,"RM180,000.00"
261613,Condominium/Apartment,94.00,sq.m,-,-,2,"RM180,000.00"
261614,Condominium/Apartment,100.00,sq.m,-,-,2,"RM180,000.00"
261615,Condominium/Apartment,94.00,sq.m,-,-,1,"RM180,000.00"
261616,Condominium/Apartment,62.62,sq.m,-,-,3,"RM45,000.00"
261617,Condominium/Apartment,62.61,sq.m,-,-,1,"RM45,000.00"
261618,Condominium/Apartment,62.61,sq.m,-,-,1,"RM42,000.00"
261619,Condominium/Apartment,69.00,sq.m,-,-,4,"RM50,000.00"
261620,Condominium/Apartment,87.00,sq.m,-,-,1,"RM118,000.00"
261621,Condominium/Apartment,88.00,sq.m,-,-,3,"RM115,000.00"


Landed rows missing Land/Parcel Area:


,Property Type,District,Mukim,Scheme Name/Area,Road Name,"Month, Year of Transaction Date",Tenure,Land/Parcel Area,Unit,Main Floor Area,Unit_2,Unit Level,Transaction Price
339142,Detached,Larut Matang,Asam Kumbang,KAMUNTING BARU TAMBAHAN (EXPO),LORONG EXPO 2,2025-10-01,Leasehold,NaN,sq.m,174,sq.m,,"RM400,000.00"


Rows missing/dash Main Floor Area by property type:


,is_landed,Property Type,missing_main_floor_rows
7,1,Town House,4251
4,1,1 - 1 1/2 Storey Terraced,9
3,1,1 - 1 1/2 Storey Semi-Detached,3
5,1,2 - 2 1/2 Storey Semi-Detached,2
6,1,Cluster House,1
0,0,Condominium/Apartment,65780
2,0,Low-Cost Flat,21190
1,0,Flat,16504


In [9]:
town_house_missing_floor = df_check.loc[
    (df_check['Property Type'] == 'Town House') &
    is_blank_or_dash(df_check[main_floor_col]),
    [
        'Property Type', 'District', 'Mukim', 'Scheme Name/Area',
        'Road Name', 'Month, Year of Transaction Date', 'Tenure',
        land_col, main_floor_col, 'Unit', 'Unit_2', 'Unit Level',
        'Transaction Price'
    ]
]

print(f"Town House rows with missing Main Floor Area: {len(town_house_missing_floor)}")
display(town_house_missing_floor)

Town House rows with missing Main Floor Area: 4251


,Property Type,District,Mukim,Scheme Name/Area,Road Name,"Month, Year of Transaction Date",Tenure,Land/Parcel Area,Main Floor Area,Unit,Unit_2,Unit Level,Transaction Price
412376,Town House,Alor Gajah,Bdr Pulau Sebang,PERUMAHAN PKN PULAU SEBANG,,2024-07-01,Freehold,108.00,-,sq.m,-,2,"RM105,000.00"
412377,Town House,Alor Gajah,Durian Tunggal,TMN ANGKASA NURI,,2022-12-01,Leasehold,67.00,-,sq.m,-,1,"RM120,000.00"
412378,Town House,Alor Gajah,Durian Tunggal,TMN ANGKASA NURI,,2023-08-01,Leasehold,68.00,-,sq.m,-,1,"RM125,000.00"
412379,Town House,Alor Gajah,Durian Tunggal,TMN ANGKASA NURI,,2024-05-01,Leasehold,68.00,-,sq.m,-,1,"RM128,000.00"
412380,Town House,Alor Gajah,Durian Tunggal,TMN ANGSA MAS,,2021-07-01,Freehold,13.00,-,sq.m,-,1,"RM100,000.00"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
416622,Town House,Timur Laut,Tanjung Tokong,TANJONG VILLA,,2022-08-01,Freehold,103.00,-,sq.m,-,1,"RM850,000.00"
416623,Town House,Timur Laut,Tanjung Tokong,TANJONG VILLA,,2024-02-01,Freehold,174.00,-,sq.m,-,1,"RM1,110,000.00"
416624,Town House,Timur Laut,Tanjung Tokong,VILLA BUNGA TELANG,,2023-07-01,Freehold,83.00,-,sq.m,-,3,"RM540,000.00"
416625,Town House,Timur Laut,Tanjung Tokong,VILLA BUNGA TELANG,,2025-01-01,Freehold,83.00,-,sq.m,-,3,"RM525,000.00"


In [10]:
# Check whether Town House rows (raw) lack Main Floor Area, and where (by District/Mukim).
town_mask = df_check['Property Type'] == 'Town House'
total_town = int(town_mask.sum())
missing_town = int(is_blank_or_dash(df_check.loc[town_mask, main_floor_col]).sum())
print(f"Town House rows: {total_town}, missing Main Floor Area (raw): {missing_town} ({missing_town/total_town*100:.2f}%)")

# Summary by District / Mukim (raw)
grp = (
    df_check.loc[town_mask]
    .groupby(['District','Mukim'], dropna=False)[main_floor_col]
    .agg(total='size', missing=lambda s: int(is_blank_or_dash(s).sum()))
    .reset_index()
)
grp['missing_pct'] = (grp['missing'] / grp['total'] * 100).round(2)

print("\nGroups with 100% missing Main Floor Area (raw):")
display(grp.loc[grp['missing_pct'] == 100].sort_values('total', ascending=False))

print("\nTop groups by missing_pct (raw):")
display(grp.sort_values(['missing_pct','total'], ascending=[False,False]).head(20))

# If processed dataframe exists, check processed 'Area' for Town House similarly
if 'df_processed' in globals():
    proc_mask = df_processed['Property Type'] == 'Town House'
    total_proc = int(proc_mask.sum())
    missing_proc = int(df_processed.loc[proc_mask, 'Area'].isna().sum())
    print(f"\nProcessed rows (Town House): {total_proc}, missing Area (processed): {missing_proc} ({missing_proc/total_proc*100:.2f}%)")

    proc_grp = (
        df_processed.loc[proc_mask]
        .groupby(['District','Mukim'], dropna=False)['Area']
        .agg(total='size', missing=lambda s: int(s.isna().sum()))
        .reset_index()
    )
    proc_grp['missing_pct'] = (proc_grp['missing'] / proc_grp['total'] * 100).round(2)

    print("\nProcessed groups with 100% missing Area:")
    display(proc_grp.loc[proc_grp['missing_pct'] == 100].sort_values('total', ascending=False))
else:
    print("\ndf_processed not available in globals()")

Town House rows: 4251, missing Main Floor Area (raw): 4251 (100.00%)

Groups with 100% missing Main Floor Area (raw):


,District,Mukim,total,missing,missing_pct
143,Sepang,Dengkil,288,288,100.0
52,Johor Bahru,Pulai,228,228,100.0
53,Johor Bahru,Tebrau,167,167,100.0
60,Klang,Kelang,122,122,100.0
71,Kuala Lumpur,Mukim Kuala Lumpur,114,114,100.0
...,...,...,...,...,...
35,Hulu Langat,Bandar Country Height,1,1,100.0
136,Seberang Perai Utara,9,1,1,100.0
140,Seberang Perai Utara,16,1,1,100.0
27,Gombak,Pekan Pengkalan Kundang,1,1,100.0



Top groups by missing_pct (raw):


,District,Mukim,total,missing,missing_pct
143,Sepang,Dengkil,288,288,100.0
52,Johor Bahru,Pulai,228,228,100.0
53,Johor Bahru,Tebrau,167,167,100.0
60,Klang,Kelang,122,122,100.0
71,Kuala Lumpur,Mukim Kuala Lumpur,114,114,100.0
44,Hulu Langat,Pekan Kajang,112,112,100.0
69,Kuala Lumpur,Mukim Batu,111,111,100.0
5,Bahagian Kuching,Bahagian Kuching,108,108,100.0
31,Hulu Langat,Bandar Ampang,100,100,100.0
78,Kuala Selangor,Ijok,99,99,100.0



Processed rows (Town House): 4251, missing Area (processed): 4251 (100.00%)

Processed groups with 100% missing Area:


,District,Mukim,total,missing,missing_pct
143,Sepang,Dengkil,288,288,100.0
52,Johor Bahru,Pulai,228,228,100.0
53,Johor Bahru,Tebrau,167,167,100.0
60,Klang,Kelang,122,122,100.0
71,Kuala Lumpur,Mukim Kuala Lumpur,114,114,100.0
...,...,...,...,...,...
35,Hulu Langat,Bandar Country Height,1,1,100.0
136,Seberang Perai Utara,9,1,1,100.0
140,Seberang Perai Utara,16,1,1,100.0
27,Gombak,Pekan Pengkalan Kundang,1,1,100.0


In [11]:
print(f"Total Town House rows: {total_town}")

Total Town House rows: 4251


In [12]:
missing_town = int(is_blank_or_dash(df_check.loc[town_mask, main_floor_col]).sum())
total_town = int(town_mask.sum())

print(f"Town House rows: {total_town}")
print(f"Town House rows with missing Main Floor Area: {missing_town}")
print(f"All Town House rows have empty Main Floor Area? {'YES' if missing_town == total_town else 'NO'}")

Town House rows: 4251
Town House rows with missing Main Floor Area: 4251
All Town House rows have empty Main Floor Area? YES


In [13]:
# Treat Town House as non-landed
if 'Town House' not in high_rise_types:
    high_rise_types = high_rise_types + ['Town House']

df_check['is_high_rise'] = df_check['Property Type'].isin(high_rise_types)
df_check['is_landed'] = (~df_check['is_high_rise']).astype(int)

high_rise_mask = df_check['is_high_rise']
landed_mask = df_check['is_landed'].eq(1)

raw_area_summary = (
    df_check
    .groupby('Property Type', dropna=False)
    .apply(completeness, include_groups=False)
    .reset_index()
)
raw_area_summary['land_missing_pct'] = (raw_area_summary['land_missing_or_dash'] / raw_area_summary['rows'] * 100).round(2)
raw_area_summary['main_floor_missing_pct'] = (raw_area_summary['main_floor_missing_or_dash'] / raw_area_summary['rows'] * 100).round(2)

print('Updated classification: Town House is now non-landed.')
print('High-rise types:', high_rise_types)
display(df_check.loc[df_check['Property Type'] == 'Town House', 'is_landed'].value_counts(dropna=False).rename_axis('is_landed').reset_index(name='rows'))
display(raw_area_summary.loc[raw_area_summary['Property Type'] == 'Town House'])

Updated classification: Town House is now non-landed.
High-rise types: ['Condominium/Apartment', 'Flat', 'Low-Cost Flat', 'Town House']


,is_landed,rows
0,0,4251


,Property Type,rows,is_landed,land_missing_or_dash,land_present,land_unique_values,main_floor_missing_or_dash,main_floor_present,main_floor_unique_values,land_missing_pct,main_floor_missing_pct
10,Town House,4251,0,0,4251,588,4251,0,0,0.0,100.0


In [14]:
property_types = df_check['Property Type'].dropna().unique().tolist()
property_types

['1 - 1 1/2 Storey Semi-Detached',
 '1 - 1 1/2 Storey Terraced',
 '2 - 2 1/2 Storey Semi-Detached',
 '2 - 2 1/2 Storey Terraced',
 'Cluster House',
 'Condominium/Apartment',
 'Detached',
 'Flat',
 'Low-Cost Flat',
 'Low-Cost House',
 'Town House']

In [15]:
# Landed properties with missing Main Floor Area
landed_missing_main = is_blank_or_dash(df_check.loc[landed_mask, main_floor_col]).sum()
landed_total = landed_mask.sum()

# High-rise properties with missing Main Floor Area
high_rise_missing_main = is_blank_or_dash(df_check.loc[high_rise_mask, main_floor_col]).sum()
high_rise_total = high_rise_mask.sum()

summary = pd.DataFrame([
    {
        'category': 'Landed',
        'total_rows': int(landed_total),
        'missing_main_floor': int(landed_missing_main),
        'missing_pct': (landed_missing_main / landed_total * 100).__round__(2),
    },
    {
        'category': 'High-rise',
        'total_rows': int(high_rise_total),
        'missing_main_floor': int(high_rise_missing_main),
        'missing_pct': (high_rise_missing_main / high_rise_total * 100).__round__(2),
    },
])

print('Main Floor Area completeness by property classification:')
display(summary)
print(f"\nDo all high-rise have missing Main Floor Area? {'YES' if high_rise_missing_main == high_rise_total else 'NO'}")

Main Floor Area completeness by property classification:


,category,total_rows,missing_main_floor,missing_pct
0,Landed,308902,15,0.0
1,High-rise,107725,107725,100.0



Do all high-rise have missing Main Floor Area? YES


In [16]:
landed_missing_floor = df_check.loc[
    landed_mask & is_blank_or_dash(df_check[main_floor_col]),
    [
        'Property Type', 'District', 'Mukim', 'Scheme Name/Area',
        'Road Name', 'Month, Year of Transaction Date', 'Tenure',
        land_col, main_floor_col, 'Unit', 'Unit_2', 'Unit Level',
        'Transaction Price'
    ]
]

print(f"Landed rows with missing Main Floor Area: {len(landed_missing_floor)}")
display(landed_missing_floor)

Landed rows with missing Main Floor Area: 15


,Property Type,District,Mukim,Scheme Name/Area,Road Name,"Month, Year of Transaction Date",Tenure,Land/Parcel Area,Main Floor Area,Unit,Unit_2,Unit Level,Transaction Price
20218,1 - 1 1/2 Storey Semi-Detached,Selama,Selama,TAMAN PALMA INDAH ( SELAMA ),JLN BUKIT BERTAM - KG TENGAH,2025-11-01,Freehold,214.0,-,sq.m,sq.m,,"RM460,000.00"
20220,1 - 1 1/2 Storey Semi-Detached,Selama,Selama,TAMAN PALMA INDAH ( SELAMA ),JLN BUKIT BERTAM - KG TENGAH,2025-12-01,Freehold,214.0,-,sq.m,sq.m,,"RM460,000.00"
20221,1 - 1 1/2 Storey Semi-Detached,Selama,Selama,TAMAN PALMA INDAH ( SELAMA ),JLN BUKIT BERTAM - KG TENGAH,2025-12-01,Freehold,219.0,-,sq.m,sq.m,,"RM461,000.00"
47194,1 - 1 1/2 Storey Terraced,Kampar,Mukim Kampar,TMN Kampar PERDANA,JLN GOPENG-KAMPAR,2026-02-01,Leasehold,255.0,-,sq.m,sq.m,,"RM300,000.00"
49136,1 - 1 1/2 Storey Terraced,Kerian,Bagan Serai,TMN SERAI SEMAMBU,LINTANG SEMAMBU 6,2025-10-01,Freehold,123.0,-,sq.m,sq.m,,"RM329,000.00"
49142,1 - 1 1/2 Storey Terraced,Kerian,Bagan Serai,TMN SERAI SEMAMBU,LINTANG SEMAMBU 7,2025-11-01,Freehold,123.0,-,sq.m,sq.m,,"RM329,000.00"
53037,1 - 1 1/2 Storey Terraced,Kinta,Ulu Kinta,TMN CHEMOR MUTIARA,HALA CHEMOR 2,2026-02-01,Leasehold,130.0,-,sq.m,sq.m,,"RM215,000.00"
65706,1 - 1 1/2 Storey Terraced,Kuala Kangsar,Sungei Siput,TMN TASIK SAUJANA,PINGGIRAN TASIK SAUJANA,2026-02-01,Leasehold,121.0,-,sq.m,sq.m,,"RM250,000.00"
86298,1 - 1 1/2 Storey Terraced,Larut Matang,Pengkalan Aor,TAMAN CHANGKAT DAMAI,JALAN CD 1,2025-12-01,Freehold,186.0,-,sq.m,sq.m,,"RM335,000.00"
87859,1 - 1 1/2 Storey Terraced,Manjung,Pengkalan Bharu,"TAMAN PANTAI SURIA, PANTAI REMIS",JALAN PANTAI SURIA 1,2026-01-01,Leasehold,149.0,-,sq.m,sq.m,,"RM289,000.00"


## Raw-data outline / preprocessing implication

1. `is_landed` is still useful because high-rise and landed rows use the area fields differently.
2. In the raw data, high-rise rows have `Land/Parcel Area` filled, and the values are not a single constant. They vary across thousands of numeric values.
3. High-rise rows have `Main Floor Area = '-'`, so the main floor area field is empty by placeholder, not by actual `NaN`.
4. Landed rows mostly have both `Land/Parcel Area` and `Main Floor Area`, but there are exceptions: one landed `Detached` row is missing `Land/Parcel Area`, and some landed rows have `Main Floor Area = '-'`.
5. For preprocessing, create `is_landed` first, then treat area columns conditionally:
   - `is_landed = 1`: use `Land/Parcel Area` as land size and `Main Floor Area` as floor size, with exception handling for dashes/blanks.
   - `is_landed = 0`: do not expect `Main Floor Area`; use/check `Land/Parcel Area` as the available size field for high-rise rows, subject to confirming its source definition.
6. Replace dash placeholders with missing values before modelling so the model does not interpret `'-'` as a category/value.


## Processed data cross-check

This checks whether the cleaned workbook preserves the same pattern seen in the raw workbook: high-rise rows have the available size in `Land`, while `Area` is empty.


In [7]:
processed_path = 'processed data/Open Transaction Data Cleaned.xlsx'
df_processed = pd.read_excel(processed_path)

processed_high_rise = df_processed['Property Type'].isin(high_rise_types)
processed_summary = pd.DataFrame([
    {
        'dataset': 'processed high-rise',
        'rows': int(processed_high_rise.sum()),
        'land_missing': int(df_processed.loc[processed_high_rise, 'Land'].isna().sum()),
        'land_present': int(df_processed.loc[processed_high_rise, 'Land'].notna().sum()),
        'land_unique_values': int(df_processed.loc[processed_high_rise, 'Land'].nunique(dropna=True)),
        'land_min': df_processed.loc[processed_high_rise, 'Land'].min(),
        'land_median': df_processed.loc[processed_high_rise, 'Land'].median(),
        'land_max': df_processed.loc[processed_high_rise, 'Land'].max(),
        'area_missing': int(df_processed.loc[processed_high_rise, 'Area'].isna().sum()),
        'area_present': int(df_processed.loc[processed_high_rise, 'Area'].notna().sum()),
        'area_unique_values': int(df_processed.loc[processed_high_rise, 'Area'].nunique(dropna=True)),
    },
    {
        'dataset': 'processed landed/other',
        'rows': int((~processed_high_rise).sum()),
        'land_missing': int(df_processed.loc[~processed_high_rise, 'Land'].isna().sum()),
        'land_present': int(df_processed.loc[~processed_high_rise, 'Land'].notna().sum()),
        'land_unique_values': int(df_processed.loc[~processed_high_rise, 'Land'].nunique(dropna=True)),
        'land_min': df_processed.loc[~processed_high_rise, 'Land'].min(),
        'land_median': df_processed.loc[~processed_high_rise, 'Land'].median(),
        'land_max': df_processed.loc[~processed_high_rise, 'Land'].max(),
        'area_missing': int(df_processed.loc[~processed_high_rise, 'Area'].isna().sum()),
        'area_present': int(df_processed.loc[~processed_high_rise, 'Area'].notna().sum()),
        'area_unique_values': int(df_processed.loc[~processed_high_rise, 'Area'].nunique(dropna=True)),
    },
])

print('Processed high-rise Land top values:')
display(df_processed.loc[processed_high_rise, 'Land'].value_counts().head(15).rename_axis('Land').reset_index(name='rows'))

processed_summary


Processed high-rise Land top values:


,Land,rows
0,61.0,3360
1,60.0,2656
2,65.0,2644
3,79.0,2474
4,70.0,2152
5,93.0,1785
6,84.0,1731
7,62.0,1466
8,64.0,1415
9,75.0,1352


,dataset,rows,land_missing,land_present,land_unique_values,land_min,land_median,land_max,area_missing,area_present,area_unique_values
0,processed high-rise,103474,0,103474,3408,12.00,79.00,5202.0,103474,0,0
1,processed landed/other,313153,1,313152,15919,8.27,143.07,1788888.0,4266,308887,826
